In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
%pip install -q kornia==0.8.0

In [4]:
from pathlib import Path
import sys


PROJECT_ROOT = Path(
    "/content/drive/MyDrive/NMA-DL2026/NMA_BrainMRI_Project"
)

CUSTOM_SRC = (
    PROJECT_ROOT
    / "notebooks"
    / "generative"
    / "custom"
    / "src"
)

if str(CUSTOM_SRC) not in sys.path:
    sys.path.insert(
        0,
        str(CUSTOM_SRC),
    )

In [5]:
import sys

sys.path.append('/content/drive/MyDrive/NMA-DL2026/NMA_BrainMRI_Project/notebooks/generative/custom/src/utils/loss')

# Now your import will work perfectly!
from loss_func import (
    calculate_loss,
    cross_entropy_loss,
    gaussian_nll_loss,
    l1_loss,
)
print("Loss imports passed.")
print("Selected training loss:", calculate_loss.__name__)

Loss imports passed.
Selected training loss: mse_loss


In [6]:
import torch


pred = torch.tensor(
    [
        1.0,
        2.0,
        3.0,
    ]
)

target = torch.tensor(
    [
        1.0,
        3.0,
        5.0,
    ]
)

loss = calculate_loss(
    pred=pred,
    target=target,
)

print("MSE loss:", loss.item())

MSE loss: 1.6666666269302368


In [7]:
expected_loss = torch.tensor(
    5.0 / 3.0
)

assert torch.isclose(
    loss,
    expected_loss,
)

print("MSE numerical test passed.")

MSE numerical test passed.


In [8]:
prediction = torch.randn(
    2,
    1,
    160,
    224,
    requires_grad=True,
)

target_noise = torch.randn_like(
    prediction
)

loss = calculate_loss(
    pred=prediction,
    target=target_noise,
)

loss.backward()

print("Loss:", loss.item())
print(
    "Gradient shape:",
    prediction.grad.shape,
)
print(
    "Gradient finite:",
    bool(
        torch.isfinite(
            prediction.grad
        ).all()
    ),
)

Loss: 1.9955939054489136
Gradient shape: torch.Size([2, 1, 160, 224])
Gradient finite: True


In [9]:
assert prediction.grad is not None

assert prediction.grad.shape == (
    2,
    1,
    160,
    224,
)

assert torch.isfinite(
    prediction.grad
).all()

print(
    "MSE backward-pass test passed."
)

MSE backward-pass test passed.


In [10]:
import torch

from base import model, noise_scheduler
from utils.loss.loss_func import calculate_loss


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = model.to(device)
model.train()

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


CustomUNet2DModel(
  (conv_in_sample): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv_in_control): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (time_proj): Timesteps()
  (time_embedding): TimestepEmbedding(
    (linear_1): Linear(in_features=32, out_features=128, bias=True)
    (act): SiLU()
    (linear_2): Linear(in_features=128, out_features=128, bias=True)
  )
  (down_blocks_sample): ModuleList(
    (0-1): 2 x DownBlock2D(
      (resnets): ModuleList(
        (0-3): 4 x ResnetBlock2D(
          (norm1): GroupNorm(32, 64, eps=1e-05, affine=True)
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (time_emb_proj): Linear(in_features=128, out_features=64, bias=True)
          (norm2): GroupNorm(32, 64, eps=1e-05, affine=True)
          (dropout): Dropout(p=0.3, inplace=False)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (nonlinearity): SiLU()
   

In [11]:
batch_size = 1

clean_sample = torch.randn(
    batch_size,
    1,
    160,
    224,
    device=device,
)

control = torch.zeros(
    batch_size,
    1,
    160,
    224,
    device=device,
)

noise = torch.randn_like(
    clean_sample
)

timesteps = torch.randint(
    low=0,
    high=noise_scheduler.config.num_train_timesteps,
    size=(batch_size,),
    device=device,
    dtype=torch.long,
)

noisy_sample = noise_scheduler.add_noise(
    original_samples=clean_sample,
    noise=noise,
    timesteps=timesteps,
)

noisy_control = noise_scheduler.add_noise(
    original_samples=control,
    noise=noise,
    timesteps=timesteps,
)

In [12]:
model.zero_grad(
    set_to_none=True
)

noise_prediction = model(
    sample=noisy_sample,
    control=noisy_control,
    timestep=timesteps,
).sample

loss = calculate_loss(
    pred=noise_prediction,
    target=noise,
)

print(
    "Predicted noise shape:",
    noise_prediction.shape,
)

print(
    "Target noise shape:",
    noise.shape,
)

print(
    "Training loss:",
    loss.item(),
)

Predicted noise shape: torch.Size([1, 1, 160, 224])
Target noise shape: torch.Size([1, 1, 160, 224])
Training loss: 0.9450341463088989


In [13]:
loss.backward()

has_gradient = any(
    parameter.grad is not None
    for parameter in model.parameters()
    if parameter.requires_grad
)

finite_gradients = all(
    torch.isfinite(
        parameter.grad
    ).all()
    for parameter in model.parameters()
    if parameter.grad is not None
)

assert noise_prediction.shape == noise.shape
assert torch.isfinite(loss)
assert has_gradient
assert finite_gradients

print(
    "Model + scheduler + loss "
    "backward test passed."
)

Model + scheduler + loss backward test passed.


In [14]:
model = model.cpu()

del clean_sample
del control
del noise
del timesteps
del noisy_sample
del noisy_control
del noise_prediction
del loss

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Temporary GPU memory released.")

Temporary GPU memory released.
